# Bicing Barcelona — Predicción del estado de las estaciones con Random Forest

**Datos:** Open Data Barcelona · 291 millones de registros (2020–2025)

---

Este notebook entrena y evalúa un modelo **Random Forest** para predecir si una estación estará **vacía**, en estado **normal** o **llena** en la próxima franja de 30 minutos.

El diseño evita el *data leakage* usando solo variables disponibles **dos franjas antes** del momento a predecir (arquitectura LAG 2):

1. **Configuración** — Bibliotecas, conexión DuckDB y parámetros globales
2. **Exploración rápida** — distribución de clases
3. **Carga de datos** — variables precalculadas con LAG(2)
4. **Entrenamiento** — Random Forest con class_weight='balanced'
5. **Evaluación** — Balanced Accuracy y Macro F1
6. **Matrices de confusión** — comparativa visual de los 3 modelos
7. **Importancia de variables** — que variables aportan más información
8. **Métricas por clase** — precision, recall, F1 por clase(vacía, normal o llena)
9. **Rendimiento por hora** — en qué horas el modelo funciona mejor o peor
10. **Resumen final** — tabla comparativa y conclusiones

### Diseño temporal (sin data leakage)

```
  t−2 (LAG 2)         t−1 (descartada)      t (OBJETIVO)
┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│  9:00–9:30   │   │  9:30–10:00  │   │ 10:00–10:30  │
│  Usada       │   │  No usada    │   │  Predecir    │
└──────────────┘   └──────────────┘   └──────────────┘
     │                                        │
     └────── features ──────────────────────┘
```

**Requisito previo:**  crear les taules `rf_features_train` i `rf_features_test` en DuckDB.

---
## 0. Configuración

Importamos las bibliotecas, conectamos a la base de datos DuckDB y definimos los parámetros globales que se utilizan a lo largo de todo el notebook.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    confusion_matrix, precision_recall_fscore_support,
)

warnings.filterwarnings('ignore')

# ── Estilo global de Matplotlib ─────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'figure.dpi':       150,
    'savefig.dpi':      150,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'font.size':        10,
    'legend.fontsize':  9,
})

# ── Colores consistentes para todo el notebook ───────────────────────────────
COLORS = {
    'rf': '#1b7332', 'ha': '#e07b00', 'sn7': '#2374ab',
    'buida': '#d62728', 'normal': '#2ca02c', 'plena': '#1f77b4',
}

# ── Carpetas de salida ───────────────────────────────────────────────────────
for carpeta in ['png', 'csv']:
    os.makedirs(carpeta, exist_ok=True)

# ── Conexión a DuckDB (solo lectura) ────────────────────────────────────────
DB_PATH = 'bicicletes.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)
print('Connexió establerta a', DB_PATH)

---
## 1. Conexión y verificación de tablas

Comprobamos que las tablas precalculadas con variables LAG(2) existen en la base de datos.


In [ ]:
# ── Verificación de tablas requeridas ────────────────────────────────────────
taules = con.execute("SHOW TABLES").fetchdf()['name'].tolist()
required = ['rf_features_train', 'rf_features_test', 'rf_nab_perfil']
missing = [t for t in required if t not in taules]

if missing:
    print(f'Falten taules: {missing}')
    print('   Executa primer: python preparar_rf_features.py')
else:
    for t in required:
        n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t}: {n:>12,} files')

---
## 2. Exploración rápida

Distribución de las tres clases (vacía, normal, llena) en el conjunto de entrenamiento (2022–2024). Como la clase `normal` es ampliamente mayoritaria, hay que utilizar métricas que penalicen este desbalance (balnced accuracy, macro-F1).


In [ ]:
# ── Distribución de clases en el conjunto de entrenamiento ──────────────────
dist = con.execute("""
    SELECT estat, COUNT(*) AS n
    FROM rf_features_train
    GROUP BY estat ORDER BY estat
""").fetchdf()

fig, ax = plt.subplots(figsize=(6, 4))
colors_pie = [COLORS.get(e, '#999') for e in dist['estat']]
ax.pie(dist['n'], labels=dist['estat'], autopct='%1.1f%%',
       colors=colors_pie, startangle=90, textprops={'fontsize': 11})
ax.set_title('Distribució de classes – Train 2022–2024', fontweight='bold')
plt.tight_layout()
plt.savefig('png/fig_distribucio_classes.png')
plt.show()

# Recuento por clase
total = dist['n'].sum()
for _, row in dist.iterrows():
    print(f"  {row['estat']:8s}: {row['n']:>10,}  ({row['n']/total*100:.1f}%)")

---
## 3. Carga de datos

Las tablas `rf_features_train` y `rf_features_test` ya contienen todas las variables precalculadas con LAG(2). El notebook solo las lee.


In [ ]:
# ── Lista de variables para el modelo ────────────────────────────────────────
FEATS = [
    # Contexto temporal
    'hora_dia', 'mitja_hora', 'dia_setmana', 'mes', 'estacio_any', 'es_festiu',
    # Estado observado en la franja t−2 (LAG 2)
    'prev_avg_bicis', 'prev_avg_anclatges', 'prev_avg_capacitat', 'prev_estat_num',
    # Tendencia t−3 → t−2
    'delta_bicis', 'delta_anclatges',
    # Perfil histórico (2022–2024)
    'nab_mig', 'nab_std', 'ratio_buida', 'ratio_plena', 'cap_mitja',
    # Cluster
    'cluster',
]
TARGET = 'estat'

# Columnas a cargar (variables + objetivo + identificadores)
cols_needed = list(dict.fromkeys(FEATS + [TARGET, 'id_estacio', 'data']))
cols_sql = ', '.join(cols_needed)

print(f'Features ({len(FEATS)}): {FEATS}')

In [ ]:
# ── Train: muestra estratificada ─────────────────────────────────────────────
SAMPLE_SIZE = 1_500_000

n_total = con.execute('SELECT COUNT(*) FROM rf_features_train').fetchone()[0]
pct = min(SAMPLE_SIZE / n_total, 1.0) * 100

df_train = con.execute(f"""
    SELECT {cols_sql}
    FROM rf_features_train
    USING SAMPLE {pct:.4f} PERCENT (bernoulli, 42)
""").fetchdf()

print(f'Train: {len(df_train):,} mostres ({pct:.1f}% de {n_total:,})')

In [ ]:
# ── Test: carga completa ─────────────────────────────────────────────────────
df_test = con.execute(f'SELECT {cols_sql} FROM rf_features_test').fetchdf()
print(f'Test: {len(df_test):,} mostres')

In [ ]:
# ── Preparar matrices X (variables) e y (objetivo) ───────────────────────────
X_train = df_train[FEATS].fillna(0).astype('float32')
y_train = df_train[TARGET]

X_test = df_test[FEATS].fillna(0).astype('float32')
y_test = df_test[TARGET]

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'\nDistribució train: {y_train.value_counts().to_dict()}')
print(f'Distribució test:  {y_test.value_counts().to_dict()}')

---
## 4. Entrenamiento del Random Forest

### Variables (todas libres de data leakage)

| Variable | Fuente | Margen temporal |
|---|---|---|
| `prev_avg_bicis`, `prev_avg_anclatges`, `prev_avg_capacitat` | Franja t−2 | 30–60 min antes |
| `prev_estat_num` | Franja t−2 | 30–60 min antess |
| `delta_bicis`, `delta_anclatges` | t−3 → t−2 | 60–90 min antes |
| `hora_dia`, `mitja_hora`, `dia_setmana`, `mes`, `estacio_any`, `es_festiu` | Calendario | Determinista |
| `nab_mig`, `nab_std`, `ratio_buida`, `ratio_plena`, `cap_mitja` | Media 2022–2024 | Estático |
| `cluster` | K-Means 2022–2024 | Estático |

El parámetro `class_weight='balanced'` compensa el desequilibrio de clases.


In [ ]:
# ── Entrenamiento ────────────────────────────────────────────────────────────
print('Entrenant Random Forest...')

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

print(f'Entrenat.  Classes: {list(rf.classes_)}')

# Liberar memoria del train (ya no lo necesitamos)
del X_train, y_train, df_train

---
## 5. Evaluación sobre test 2025

Comparamos el Random Forest con los dos baselines. La balanced accuracy penaliza los modelos que solo aciertan la clase mayoritaria.


In [ ]:
# ── Predicción del Random Forest ─────────────────────────────────────────────
y_pred_rf = rf.predict(X_test)

ba_rf  = balanced_accuracy_score(y_test, y_pred_rf)
mf1_rf = f1_score(y_test, y_pred_rf, average='macro')

print(f'Random Forest (LAG 2, 30 min marge):')
print(f'  Balanced Accuracy: {ba_rf*100:.1f}%')
print(f'  Macro F1-Score:    {mf1_rf*100:.1f}%')

In [ ]:
# ── Cargar resultados de los baselines (precalculados en DuckDB) ───────────
ba_df = con.execute("""
    SELECT baseline,
           balanced_accuracy_pct / 100.0 AS ba,
           macro_f1_pct          / 100.0 AS mf1
    FROM balanced_accuracy
""").fetchdf()

metrics_all = {
    'Random Forest':     {'ba': ba_rf,  'mf1': mf1_rf},
    'Historical Avg':    {'ba': ba_df.loc[ba_df.baseline=='ha', 'ba'].values[0],
                          'mf1': ba_df.loc[ba_df.baseline=='ha', 'mf1'].values[0]},
    'Seasonal Naive 7d': {'ba': ba_df.loc[ba_df.baseline=='sn7','ba'].values[0],
                          'mf1': ba_df.loc[ba_df.baseline=='sn7','mf1'].values[0]},
}

# Tabla comparativa
print(f'  {"Model":20s}  {"Bal.Acc.":>8s}   {"Macro-F1":>8s}')
print(f'  {"─"*42}')
for name, m in metrics_all.items():
    print(f'  {name:20s}  {m["ba"]*100:7.1f}%   {m["mf1"]*100:7.1f}%')

In [ ]:
# ── Gráfico comparativo: Balanced Accuracy y Macro F1 ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
models = list(metrics_all.keys())
colors_m = [COLORS['rf'], COLORS['ha'], COLORS['sn7']]
ba_vals = [metrics_all[m]['ba']*100 for m in models]
f1_vals = [metrics_all[m]['mf1']*100 for m in models]

for ax, vals, title in zip(axes, [ba_vals, f1_vals],
                           ['Balanced Accuracy (%)', 'Macro F1-Score (%)']):
    bars = ax.bar(models, vals, color=colors_m, edgecolor='white', width=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.1f}%', ha='center', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.axhline(33.3, color='grey', ls='--', lw=0.8, label='Atzar (33%)')
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Comparativa – Predicció franja següent (LAG 2, 30 min marge)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('png/prediccions_balanced_accuracy.png')
plt.show()

---
## 6. Matrices de confusión

Matrices normalizadas por fila: cada casilla muestra el porcentaje de franjas de una clase real que han sido predichas como cada clase. La diagonal principal son los aciertos.

> **Diapostiva 10** (inferior)

In [ ]:
# ── Unir predicciones RF con baselines (precalculadas) ───────────────────────
eval_bl = con.execute("""
    SELECT id_estacio, data, hora_dia, mitja_hora,
           estat_real, ha_pred, sn7_pred
    FROM eval_baselines
""").fetchdf()

df_test['data'] = pd.to_datetime(df_test['data'])
eval_bl['data'] = pd.to_datetime(eval_bl['data'])

merged = df_test.merge(
    eval_bl, on=['id_estacio','data','hora_dia','mitja_hora'], how='inner'
)

# Filtrar: solo filas con estado válido y predicciones baseline no nulas
merged = merged[
    merged['estat'].isin(['buida','normal','plena'])
    & merged['ha_pred'].notna()
    & merged['sn7_pred'].notna()
].copy()

merged['ha_pred']  = merged['ha_pred'].astype(str)
merged['sn7_pred'] = merged['sn7_pred'].astype(str)

# Vectores de predicción sobre el conjunto común
y_real = merged['estat']
y_ha   = merged['ha_pred']
y_sn7  = merged['sn7_pred']
y_rfm  = rf.predict(merged[FEATS].fillna(0).astype('float32'))

print(f'Files comparables: {len(merged):,}')
print(f'RF BA (merged):  {balanced_accuracy_score(y_real, y_rfm)*100:.1f}%')
print(f'HA BA (merged):  {balanced_accuracy_score(y_real, y_ha)*100:.1f}%')
print(f'SN7 BA (merged): {balanced_accuracy_score(y_real, y_sn7)*100:.1f}%')

In [ ]:
# ── Matrices de confusión normalizadas por fila (3 modelos) ─────────────────
classes_ord = ['buida', 'normal', 'plena']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
predictions = {
    'Random Forest': y_rfm,
    'Historical Average': y_ha,
    'Seasonal Naive 7d': y_sn7,
}

for ax, (name, yp) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_real, yp, labels=classes_ord, normalize='true') * 100
    sns.heatmap(cm, annot=True, fmt='.1f', cmap='YlGnBu',
                xticklabels=classes_ord, yticklabels=classes_ord,
                ax=ax, vmin=0, vmax=100, cbar_kws={'label': '%'})
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_ylabel('Realitat' if ax == axes[0] else '')
    ax.set_xlabel('Predicció')

plt.suptitle('Matrius de confusió – test 2025 (normalitzades per fila)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('png/prediccions_matrices_confusio.png')
plt.show()

---
## 7. Importancia de las variables

¿Qué variables aportan más información al Random Forest? El gráfico muestra la importancia relativa de cada variable según la reducción media de impureza (Gini).

> **Diapositiva 11** de la presentación


In [ ]:
# ── Importancias del modelo ──────────────────────────────────────────────────
importances = rf.feature_importances_

# Ordenar de mayor a menor
order = np.argsort(importances)[::-1]
sorted_names = [FEATS[i] for i in order]
sorted_vals  = importances[order]

# Nombres para el gráfico
NOMS = {
    'prev_avg_bicis':     'Bicis disp. (t−2)',
    'prev_avg_anclatges': 'Anclatges disp. (t−2)',
    'prev_avg_capacitat': 'Capacitat (t−2)',
    'prev_estat_num':     'Estat (t−2)',
    'delta_bicis':        'Tendència bicis (t−3→t−2)',
    'delta_anclatges':    'Tendència anclatges (t−3→t−2)',
    'nab_mig':            'NAB mig històric',
    'nab_std':            'Variabilitat NAB',
    'ratio_buida':        'Freq. hist. buida',
    'ratio_plena':        'Freq. hist. plena',
    'cap_mitja':          'Capacitat històrica',
    'hora_dia':           'Hora del dia',
    'mitja_hora':         '1a/2a meitat hora',
    'dia_setmana':        'Dia de la setmana',
    'mes':                'Mes',
    'estacio_any':        "Estació de l'any",
    'es_festiu':          'Festiu / cap setmana',
    'cluster':            'Cluster estació',
}
sorted_labels = [NOMS.get(n, n) for n in sorted_names]

# Gráfico de barras horizontales
fig, ax = plt.subplots(figsize=(9, 6))
n_feats = len(sorted_vals)
colors_fi = plt.cm.RdYlGn(np.linspace(0.3, 0.9, n_feats)[::-1])

# barh pinta de abajo a arriba → invertimos para tener el más importante arriba
bars = ax.barh(range(n_feats), sorted_vals[::-1]*100,
               color=colors_fi, edgecolor='white')
ax.set_yticks(range(n_feats))
ax.set_yticklabels(sorted_labels[::-1])
ax.set_xlabel('Importància (%)')
ax.set_title('Importància de les variables – Random Forest (LAG 2)',
             fontweight='bold')

for bar, val in zip(bars, sorted_vals[::-1]*100):
    ax.text(val + 0.2, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('png/prediccions_variables_importants.png')
plt.show()

---
## 8. Métricas por clase

Precision, recall y F1-score para cada clase y modelo.


In [ ]:
# ── Precision, Recall y F1 por clase y modelo (sobre el conjunto merged) ────
classes_ord = ['buida', 'normal', 'plena']

models_pred = {
    'Random Forest': y_rfm,
    'Historical Avg (HA)': y_ha,
    'Seasonal Naive 7d (SN7)': y_sn7,
}

all_metrics = []
model_arrays = {}

for model_name, y_pred_model in models_pred.items():
    p, r, f, _ = precision_recall_fscore_support(
        y_real, y_pred_model, labels=classes_ord, zero_division=0
    )
    model_arrays[model_name] = {'p': p, 'r': r, 'f': f}
    for j, classe in enumerate(classes_ord):
        all_metrics.append({
            'Model': model_name,
            'Classe': classe,
            'Precision (%)': round(p[j]*100, 1),
            'Recall (%)':    round(r[j]*100, 1),
            'F1 (%)':        round(f[j]*100, 1),
        })

df_all_metrics = pd.DataFrame(all_metrics)

# Mostrar métricas por modelo
for model_name in models_pred:
    sub = df_all_metrics[df_all_metrics.Model == model_name].drop(columns='Model')
    print(f'\n--- {model_name} ---')
    display(sub.reset_index(drop=True))

# Tabla resumen de F1 por clase
print('\n--- F1 per classe i model ---')
pivot = df_all_metrics.pivot(index='Classe', columns='Model', values='F1 (%)')
pivot = pivot[list(models_pred.keys())]
display(pivot)

In [ ]:
# ── Gráfico F1 por clase y modelo ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(classes_ord))
w = 0.25

model_names = list(models_pred.keys())
model_colors = [COLORS['rf'], COLORS['ha'], COLORS['sn7']]

for k, (mname, mcolor) in enumerate(zip(model_names, model_colors)):
    f1_vals = model_arrays[mname]['f']
    offset = (k - 1) * w
    bars = ax.bar(x + offset, f1_vals*100, w, label=mname, color=mcolor)
    for bar, val in zip(bars, f1_vals*100):
        if val > 3:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                    f'{val:.0f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(classes_ord)
ax.set_ylabel('F1-Score (%)')
ax.set_title('F1-Score per classe i model (test 2025)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('png/prediccions_metriques.png')
plt.show()

---
## 9. Rendimiento por hora del día

¿A qué horas funciona mejor o peor el modelo? Las franjas de punta (mañana y tarde) son las más difíciles porque concentran los cambios de estado.

> **Diapositiva 10** de la presentación

In [ ]:
# ── Balanced Accuracy por hora para cada modelo ──────────────────────────────
merged_eval = merged.assign(pred_rf=y_rfm)

resultats_hora = []
for hora, grp in merged_eval.groupby('hora_dia'):
    if len(grp) < 100:
        continue
    resultats_hora.append({
        'hora': hora,
        'RF':  balanced_accuracy_score(grp['estat'], grp['pred_rf'])*100,
        'HA':  balanced_accuracy_score(grp['estat'], grp['ha_pred'])*100,
        'SN7': balanced_accuracy_score(grp['estat'], grp['sn7_pred'])*100,
    })
df_hora = pd.DataFrame(resultats_hora)

# Gráfico de líneas con franjas de punta
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_hora.hora, df_hora.RF,  'o-', color=COLORS['rf'],  lw=2,   label='Random Forest')
ax.plot(df_hora.hora, df_hora.HA,  's-', color=COLORS['ha'],  lw=1.5, label='Historical Avg')
ax.plot(df_hora.hora, df_hora.SN7, '^-', color=COLORS['sn7'], lw=1.5, label='Seasonal Naive 7d')
ax.axhline(33.3, color='grey', ls='--', lw=0.8)
ax.axvspan(7, 10,  alpha=0.08, color='red',  label='Punta matí')
ax.axvspan(17, 20, alpha=0.08, color='blue', label='Punta tarda')
ax.set_xlabel('Hora del dia')
ax.set_ylabel('Balanced Accuracy (%)')
ax.set_title('Rendiment per hora – Test 2025 (LAG 2)', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xticks(range(0, 24))
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('png/prediccions_rendiment_hora.png')
plt.show()

---
## 10. Resumen final

Tabla comparativa de los tres modelos y conclusiones sobre la viabilidad de la predicción.


In [ ]:
# ── Tabla resumen ────────────────────────────────────────────────────────────
ba_rf_f  = balanced_accuracy_score(y_real, y_rfm) * 100
ba_ha_f  = balanced_accuracy_score(y_real, y_ha)  * 100
ba_sn7_f = balanced_accuracy_score(y_real, y_sn7) * 100

resum = pd.DataFrame({
    'Model': ['Random Forest (LAG 2)', 'Seasonal Naive 7d', 'Historical Average'],
    'Balanced Accuracy (%)': [ba_rf_f, ba_sn7_f, ba_ha_f],
    'Macro F1 (%)': [
        f1_score(y_real, y_rfm, average='macro', labels=classes_ord)*100,
        f1_score(y_real, y_sn7, average='macro', labels=classes_ord)*100,
        f1_score(y_real, y_ha,  average='macro', labels=classes_ord)*100,
    ],
}).round(1)

print('='*60)
print('  RESUM FINAL')
print('='*60)
display(resum)
print(f'\nMillora RF vs Historical Avg:  +{ba_rf_f - ba_ha_f:.1f} pp')
print(f'Millora RF vs Seasonal Naive:  +{ba_rf_f - ba_sn7_f:.1f} pp')
print()
print('Disseny temporal:')
print('  - Les features observacionals provenen de la franja t−2 (LAG 2)')
print("  - La franja t−1 NO s'utilitza → 30 min de marge per actuar")
print('  - Les features històriques (NAB, ratios) són de 2022–2024')
print('  - Cap feature conté informació de la franja objectiu')
print(f"\nAquest model es podria executar en producció a les XX:00 i XX:30")
print(f"per predir l'estat de les estacions 30 min després.")

### 10.1 Exportación CSV — datos de predicción


In [ ]:
# ── CSV 1: Métricas por clase y modelo ───────────────────────────────────────
df_all_metrics.to_csv('csv/prediccions_metriques_classe.csv', index=False, encoding='utf-8-sig')

# CSV 2: Resumen comparativo
resum.to_csv('csv/prediccions_resum.csv', index=False, encoding='utf-8-sig')

# CSV 3: Rendimiento por hora
df_hora.to_csv('csv/prediccions_rendiment_hora.csv', index=False, encoding='utf-8-sig')

print('CSV exportats a csv/')

---
## 11. Cierre

Cerramos la conexión y listamos todos los archivos generados.


In [ ]:
con.close()
print('Connexió a DuckDB tancada.\n')

print('=== Fitxers generats ===')
for carpeta in ['png', 'csv']:
    fitxers = sorted(glob.glob(f'{carpeta}/*'))
    if fitxers:
        print(f'\n{carpeta.upper()}/  ({len(fitxers)} fitxers)')
        for f in fitxers:
            print(f'  {f}  ({os.path.getsize(f)/1024:.0f} KB)')